# Notebook 06
# Advanced Community Ecology and Multivariate Statistical Analysis

## Omo Forest Reserve Ecological Informatics Project

---

### Objectives

This notebook performs advanced community ecology analyses using the validated
community matrix generated in Notebook 05.

The analyses include:

- Data loading and validation
- Alpha diversity assessment
- Statistical comparison of diversity among ecological zones
- Beta diversity analysis
- Hierarchical cluster analysis
- Principal Component Analysis (PCA)
- Non-metric Multidimensional Scaling (NMDS)
- PERMANOVA
- Indicator Species Analysis
- Publication-ready visualizations
- Export of tables and figures

---

### Input Files

- omo_community_matrix.csv
- omo_plot_metadata.csv
- omo_species_metadata.csv

---

### Outputs

Tables/
Figures/

---

Author:
Peter Ugege et al.

Project:
Ecological Informatics Analysis of Tree Community Structure in Omo Forest Reserve


# Table of Contents

1. Import Libraries
2. Configure Plotting
3. Load Input Data
4. Validate Input Data
5. Alpha Diversity
6. Diversity Comparisons
7. Beta Diversity
8. Cluster Analysis
9. Principal Component Analysis
10. NMDS
11. PERMANOVA
12. Indicator Species Analysis
13. Export Tables
14. Export Figures
15. Notebook Summary

In [39]:
# ==========================================================
# Import Required Libraries
# ==========================================================

import warnings
warnings.filterwarnings("ignore")

import os
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import kruskal
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import pdist

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from skbio.diversity import alpha_diversity
from skbio.diversity import beta_diversity
from skbio.stats.ordination import pcoa

from skbio.stats.distance import permanova

from sklearn.manifold import MDS

from matplotlib.ticker import MaxNLocator

In [40]:
# ==========================================================
# Publication Figure Style
# ==========================================================

sns.set_theme(style="whitegrid")

plt.rcParams.update({

    "figure.figsize": (8,6),
    "figure.dpi":300,

    "font.size":11,
    "axes.titlesize":13,
    "axes.labelsize":12,

    "legend.fontsize":10

})

In [ ]:
project_root = Path.cwd().parent

OUTPUT_DIR = project_root / "outputs"

TABLE_DIR = OUTPUT_DIR / "tables"

FIGURE_DIR = OUTPUT_DIR / "figures"

In [41]:
# ==========================================================
# Output Directories
# ==========================================================

project_root = Path.cwd().parent

OUTPUT_DIR = project_root / "outputs"

TABLE_DIR = OUTPUT_DIR / "tables"

FIGURE_DIR = OUTPUT_DIR / "figures"

TABLE_DIR.mkdir(exist_ok=True)

FIGURE_DIR.mkdir(exist_ok=True)

In [42]:
# ==========================================================
# Load Input Data
# ==========================================================

community = pd.read_csv(TABLE_DIR / "omo_community_matrix.csv")

plots = pd.read_csv(TABLE_DIR / "omo_plot_metadata.csv")

species = pd.read_csv(TABLE_DIR / "omo_species_metadata.csv")

print("Community Matrix:", community.shape)

print("Plot Metadata:", plots.shape)

print("Species Metadata:", species.shape)

Community Matrix: (90, 151)
Plot Metadata: (90, 3)
Species Metadata: (148, 1)


In [43]:
# ==========================================================
# Validate Community Matrix
# ==========================================================

print("="*60)

print("Community Matrix Validation")

print("="*60)

print()

print("Duplicate plots:", community.iloc[:,0].duplicated().sum())

print("Missing values:", community.isna().sum().sum())

print()

print("Negative abundances:", (community.iloc[:,1:] < 0).sum().sum())

print()

print("Total Individuals:",
      community.iloc[:,1:].values.sum())

print()

print("Species:",
      community.shape[1]-1)

print()

print("Plots:",
      community.shape[0])

Community Matrix Validation

Duplicate plots: 0
Missing values: 0



TypeError: '<' not supported between instances of 'str' and 'int'

In [44]:
community.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90 entries, 0 to 89
Columns: 151 entries, Plot_ID to 18
dtypes: int64(148), object(3)
memory usage: 106.3+ KB


In [45]:
community.head()

,Plot_ID,Zone,Habitat,Amphimas tetracoides,Annonidium manni,Anthonotha macrophylla,Blighia Sapida,Celtis zenkeri,Cleistopholis patens,Diospyros dendo,...,Carica papaya,Chrysophyllum albidum,Citrus sinenses,Dracena manni,Elaeis guineense,Ficus mucuso,Musa sapiensis,Raulvolfia vomitoria,Trema orientalis,18
0,CmP1,Core,Major River,1,2,0,0,4,1,3,...,0,0,0,0,0,0,0,0,0,0
1,CmP2,Core,Major River,0,2,0,1,0,0,1,...,0,0,0,0,0,0,0,0,0,0
2,CmP3,Core,Major River,0,1,1,0,0,0,2,...,0,0,0,0,0,0,0,0,0,0
3,CmP4,Core,Major River,0,0,0,0,0,1,5,...,0,0,0,0,0,0,0,0,0,0
4,CmP5,Core,Major River,0,0,0,0,0,0,7,...,0,0,0,0,0,0,0,0,0,0


In [46]:
community.columns.tolist()

['Plot_ID',
 'Zone',
 'Habitat',
 'Amphimas tetracoides',
 'Annonidium manni',
 'Anthonotha macrophylla',
 'Blighia Sapida',
 'Celtis zenkeri',
 'Cleistopholis patens',
 'Diospyros dendo',
 'Drypetes species',
 'Funtumia elastica',
 'Homalium aylmeri',
 'irvinga gabonensis',
 'Lecaniodiscus cupanioides',
 'Lonchocarpus cilliata',
 'Margaritaria discoidea',
 'Millettia thoningii',
 'Monodora tenuifolia',
 'Nesodogornia papaverifera',
 'Picralima nitida',
 'Pycnanthus angolensis',
 'Ricinodendron heudelotii',
 'Scotellia coricea',
 'Sterculia rhinopetala',
 'Strombosia  pustulata',
 'Trichilia gilgiana',
 'Trichilia monadelpha',
 'Uapaca heudelotii',
 '26 species',
 'Alstonia  boonei',
 'Blighia sapida',
 'Bombax buonopozene',
 'Bosquei angolensis',
 'Cleistopholis Patens',
 'Dialium guinensis',
 'Ficus exasperata',
 'Grewia pubescens',
 'Homalium alymeri',
 'Musanga cecropiodes',
 'Nesodogornia papaverivera',
 'Pavetta corymbosa',
 'Ricinodendron heudeloti',
 'Sterculia tragacantha',
 '

In [47]:
invalid_cols = [
    c for c in community.columns
    if c.strip().isdigit() or "species" in c.lower()
]

invalid_cols

['Drypetes species',
 '26 species',
 '24',
 'Cola Species',
 '25',
 '33',
 '28',
 '27',
 '14',
 '18']